# Lab 6 — The importance of plotting

Two datasets whose **summary statistics are (almost) identical** but whose **shapes are completely different**: Anscombe's quartet (`ans.csv`) and the Datasaurus dozen (`ans2.tsv`). The tables below are computed from the data and the scatter plots show why the numbers alone are misleading.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', lambda v: f'{v:.3f}')

## Task 1 — Anscombe's quartet (`ans.csv`)

Four (x, y) datasets. For each we compute the mean and **sample** variance of x and y, the correlation, the least-squares line `y = a + b·x`, and its coefficient of determination R².

In [2]:
ans = pd.read_csv('reference/ans.csv', index_col=0)
rows = []
for i in [1, 2, 3, 4]:
    x = ans[f'x{i}'].to_numpy(); y = ans[f'y{i}'].to_numpy()
    b, a, r, p, se = stats.linregress(x, y)
    rows.append({
        'dataset': f'set {i}',
        'mean x': x.mean(),
        'sample var x': x.var(ddof=1),
        'mean y': y.mean(),
        'sample var y': y.var(ddof=1),
        'corr(x,y)': r,
        'reg a (intercept)': a,
        'reg b (slope)': b,
        'R^2': r**2,
    })
anscombe_stats = pd.DataFrame(rows).set_index('dataset')
anscombe_stats.round(3)

,mean x,sample var x,mean y,sample var y,"corr(x,y)",reg a (intercept),reg b (slope),R^2
dataset,,,,,,,,
set 1,9.000,11.000,7.501,4.127,0.816,3.000,0.500,0.667
set 2,9.000,11.000,7.501,4.128,0.816,3.001,0.500,0.666
set 3,9.000,11.000,7.500,4.123,0.816,3.002,0.500,0.666
set 4,9.000,11.000,7.501,4.123,0.817,3.002,0.500,0.667


The filled table (values rounded to ~2-3 places):

| Property | Value | Accuracy |
| --- | --- | --- |
| Mean of x | 9.0 | exact for all four sets |
| Sample variance of x | 11.0 | exact for all four sets |
| Mean of y | 7.50 | to 2 places (7.50, 7.50, 7.50, 7.50) |
| Sample variance of y | 4.13 | to 2 places (4.127 each) |
| Correlation between x and y | 0.816 | to 3 places |
| Linear regression line | y = 3.00 + 0.500·x | to 2-3 places |
| R² of the regression | 0.667 | to 3 places |

All four datasets share the same statistics — yet they look nothing alike:

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
xs = np.linspace(3, 20, 100)
for i, ax in zip([1, 2, 3, 4], axes.flat):
    x = ans[f'x{i}']; y = ans[f'y{i}']
    b, a, r, p, se = stats.linregress(x, y)
    ax.scatter(x, y, color='tab:blue', s=40, zorder=3)
    ax.plot(xs, a + b*xs, color='tab:red', lw=1.5)
    ax.set_title(f'Set {i}: y = {a:.2f} + {b:.2f}x,  r = {r:.3f}')
    ax.grid(True, alpha=0.3)
fig.suptitle("Anscombe's quartet — identical stats, different shapes", fontsize=14, fontweight='bold')
fig.supxlabel('x'); fig.supylabel('y')
plt.tight_layout(); plt.show()

/tmp/ipykernel_210115/4021269670.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Reading:** set 1 is a genuine linear relationship; set 2 is a clean parabola (a line is the wrong model); set 3 is perfectly linear except for one outlier that drags the slope; set 4 is a vertical stack plus a single high-leverage point that alone defines the fitted line.

## Task 2 — Datasaurus dozen (`ans2.tsv`)

Thirteen datasets of 142 points each. For every dataset we compute mean x, mean y, SD x, SD y and the correlation, rounded to 3 places.

In [4]:
dino = pd.read_csv('reference/ans2.tsv', sep='\t')
g = dino.groupby('dataset')
summary = pd.DataFrame({
    'mean x': g['x'].mean(),
    'mean y': g['y'].mean(),
    'sd x': g['x'].std(ddof=1),
    'sd y': g['y'].std(ddof=1),
    'corr': g.apply(lambda d: d['x'].corr(d['y']), include_groups=False),
}).round(3)
summary

,mean x,mean y,sd x,sd y,corr
dataset,,,,,
a,54.266,47.835,16.770,26.940,-0.064
bullseye,54.269,47.831,16.769,26.936,-0.069
circle,54.267,47.838,16.760,26.930,-0.068
d,54.263,47.832,16.765,26.935,-0.064
dots,54.260,47.840,16.768,26.930,-0.060
h,54.261,47.830,16.766,26.940,-0.062
high,54.269,47.835,16.767,26.940,-0.069
s,54.267,47.840,16.769,26.930,-0.063
slant_down,54.268,47.836,16.767,26.936,-0.069


Every dataset has essentially the same mean x ≈ 54.3, mean y ≈ 47.8, SD x ≈ 16.8, SD y ≈ 26.9 and corr ≈ −0.06. Now the scatter plots, one subplot per dataset (same panel idea as Task 5 in Lab 5):

In [5]:
names = sorted(dino['dataset'].unique())
fig, axes = plt.subplots(3, 5, figsize=(15, 9), sharex=True, sharey=True)
for name, ax in zip(names, axes.flat):
    d = dino[dino['dataset'] == name]
    ax.scatter(d['x'], d['y'], s=8, color='tab:green')
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
for ax in axes.flat[len(names):]:
    ax.axis('off')
fig.suptitle('Datasaurus dozen — same summary statistics, wildly different shapes', fontsize=14, fontweight='bold')
fig.supxlabel('x'); fig.supylabel('y')
plt.tight_layout(); plt.show()

/tmp/ipykernel_210115/1493965101.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### What do "d" and "s" stand for?

The 13 datasets are the **Datasaurus dozen**. The short labels are shape names:

- **`d` = *dino*** — the points draw a **dinosaur** (the original *Datasaurus*).
- **`s` = *star*** — the points draw a **five-pointed star**.

(The rest follow the same convention: `a`=away, `h`=h_lines, `v`=v_lines, `x`=x_shape, `high`=high_lines, plus dots, circle, bullseye, slant_up, slant_down, wide_lines.)

**Conclusion:** identical means, standard deviations and correlations tell you almost nothing about the data — you must plot it. See [*Same Stats, Different Graphs*](https://www.autodeskresearch.com/publications/samestats) and [Anscombe's quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet).